In [6]:
from langchain_ollama import ChatOllama,OllamaEmbeddings

llm = ChatOllama(
    model="minimax-m3:cloud",
    temperature=0
)

In [45]:
import os
import json
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

In [7]:
embeddings = OllamaEmbeddings(model="mxbai-embed-large:latest")

In [2]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.store.memory import InMemoryStore
from langmem import create_manage_memory_tool,create_search_memory_tool

In [28]:
# set up the store
store=InMemoryStore(
    index={
        "embed":embeddings,
        "dims": 1536,
        
    }
)

In [29]:
# create an agent with memory capabilities
agent=create_agent(
    model=llm,
    tools=[
        # memory tools use langGraph's BaseStore for persistence(4)
        create_manage_memory_tool(namespace=("memories",)),
        create_search_memory_tool(namespace=("memories",)),
    ],
    store=store,
)

In [38]:
# store a new memory
agent.invoke(
    {"messages":[{"role":"user",
                  "content":"Remember my favorite food is pasta"
                  
                  }]}
)

{'messages': [HumanMessage(content='Remember my favorite food is pasta', additional_kwargs={}, response_metadata={}, id='f1ac1ac4-3637-40d6-9eb0-e6e064041bf5'),
  AIMessage(content="I'll remember that your favorite food is pasta!", additional_kwargs={}, response_metadata={'model': 'minimax-m3', 'created_at': '2026-07-15T06:46:36.719331231Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2728702069, 'load_duration': None, 'prompt_eval_count': 601, 'prompt_eval_duration': None, 'eval_count': 81, 'eval_duration': None, 'logprobs': None, 'model_name': 'minimax-m3', 'model_provider': 'ollama'}, id='lc_run--019f6486-f042-7292-9972-d4bb3ded10a3-0', tool_calls=[{'name': 'manage_memory', 'args': {'content': "User's favorite food is pasta", 'action': 'create'}, 'id': 'a5301f15-a1ef-4191-a297-0f28c275f260', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 601, 'output_tokens': 81, 'total_tokens': 682}),
  ToolMessage(content='created memory 8d4f586d-34f9-4e38

In [39]:
# Retrieve the stored memory
response=agent.invoke(
    {"messages":[{"role":"user",
                  "content":"what is my favourite food"
                  }]}
)

In [40]:
from pprint import pprint
pprint(response["messages"][-1].content)

('Your favourite food is **pasta**! 🍝 Would you like recommendations for new '
 'pasta recipes to try?')


#### Health coach

In [41]:
store1=InMemoryStore()

In [42]:
# create memory tools
manage_memory=create_manage_memory_tool(
    namespace=("health_coach","user123","memories"),
    store=store1,
)

In [43]:
# create search tools
search_memory=create_search_memory_tool(
    namespace=("health_coach","user123","memories"),
    store=store,
)

In [44]:
# System prompt for the health coach
SYSTEM_PROMPT = """
You are a personal health coach with access to memory tools. Your goal is to help users achieve their fitness goals
by providing personalized advice and tracking their progress over time.

You have two special abilities:
1. You can search past memories about the user to provide personalized advice
2. You can store new information about the user for future reference

When appropriate, you will:
- Search memory for user context (diet, fitness level, goals, etc.)
- Store important new information (changes in weight, new goals, etc.)
- Reference past context to provide continuity

Always be supportive, encouraging, and provide actionable advice.
"""

In [46]:
def run_interactive_health_coach():
    """
    Interactive console interface for the health coach with memory capabilities
    """
    print("\n" + "=" * 80)
    print("🏃‍♂️ WELCOME TO YOUR PERSONAL HEALTH COACH 🏋️‍♀️")
    print("=" * 80)
    print(
        "I'm your AI health coach with memory. I can provide personalized fitness advice"
    )
    print("and will remember details about your fitness journey between conversations.")
    print("\nType 'exit', 'quit', or 'bye' to end our conversation.\n")

    # User ID - in a real app, this would be linked to user accounts
    user_id = "user123"

    # Initialize conversation history
    conversation_history = []

    # Start the conversation loop
    while True:
        # Get user input
        user_message = input("\nYou: ")

        # Check if user wants to exit
        if user_message.lower() in ["exit", "quit", "bye"]:
            print(
                "\nThank you for chatting with your health coach! Stay healthy and keep moving!"
            )
            break

        # Search memory for relevant context
        memory_results = "[]"
        try:
            # Create a search query based on user input and conversation history
            search_query = user_message
            if len(search_query) > 100:
                search_query = search_query[:100]  # Limit query length

            print("\n🔍 Searching memory for relevant context...")
            memory_results = search_memory.invoke({"query": search_query})

            # Parse and pretty-print memory results for debugging
            if memory_results and memory_results != "[]":
                try:
                    memory_items = json.loads(memory_results)
                    if memory_items:
                        print(f"📚 Found {len(memory_items)} relevant memories:")
                        for idx, item in enumerate(memory_items):
                            if "value" in item and "content" in item["value"]:
                                print(
                                    f"  • Memory {idx+1}: {item['value']['content'][:100]}..."
                                )
                    else:
                        print("📭 No relevant memories found.")
                except:
                    print(f"⚠️ Memory format: {memory_results[:100]}...")
            else:
                print("📭 No previous memories found.")
        except Exception as e:
            print(f"⚠️ Memory search error: {str(e)}")

        # Add to conversation history
        conversation_history.append(user_message)

        # Create context for the model
        context = (
            "\n".join(conversation_history[-3:])
            if len(conversation_history) > 1
            else user_message
        )

        # Generate response with context
        messages = [
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(
                content=f"""
User message: {user_message}

Previous conversation context: {context}

Memory search results: {memory_results}

Respond to the user with personalized health coaching advice based on their message and any relevant information from memory.

Also, identify if there's new important information about the user that should be stored in memory.
Don't mention the memory system directly to the user - they should just experience personalized advice.
"""
            ),
        ]

        # Get response from model
        print("\n⏳ Thinking...")
        response = llm.invoke(messages)

        # Process response to extract potential memory updates
        coach_response = response.content
        memory_to_store = None

        # Check if the response contains a clear indication of memory content
        if (
            "MEMORY:" in coach_response
            or "STORE IN MEMORY:" in coach_response
            or "REMEMBER:" in coach_response
        ):
            try:
                # Try to extract memory section - assumes model might format its response with a designated memory section
                for marker in ["MEMORY:", "STORE IN MEMORY:", "REMEMBER:"]:
                    if marker in coach_response:
                        parts = coach_response.split(marker, 1)
                        if len(parts) > 1:
                            # Extract the memory part and clean up the response
                            memory_section = parts[1].split("\n\n", 1)[0].strip()
                            coach_response = coach_response.replace(
                                marker + memory_section, ""
                            ).strip()
                            memory_to_store = memory_section
                            break
            except:
                # If extraction fails, create a general memory
                memory_to_store = f"User message: {user_message}"
        else:
            # Create an automatic memory from this interaction
            memory_to_store = extract_key_information(user_message)

        # Store memory if we have something to store
        if memory_to_store:
            try:
                print("\n💾 Storing new information in memory...")
                manage_memory.invoke({"content": memory_to_store})
                print(
                    f"✅ Stored: {memory_to_store[:50]}..."
                    if len(memory_to_store) > 50
                    else f"✅ Stored: {memory_to_store}"
                )
            except Exception as e:
                print(f"⚠️ Memory storage error: {str(e)}")

        # Clean up response to remove any artifacts and display to user
        coach_response = clean_response(coach_response)
        print(f"\nCoach: {coach_response}")


def extract_key_information(user_message):
    """
    Automatically extract key health/fitness information from user messages
    when no explicit memory instruction is found
    """
    # Simple extraction for demonstration - in a real system, use NLP for better extraction
    memory = f"User shared: {user_message}"
    return memory


def clean_response(response):
    """Clean up the response to remove any system artifacts"""
    # Remove common formatting markers
    for marker in ["RESPONSE:", "USER RESPONSE:", "COACH:"]:
        if response.startswith(marker):
            response = response[len(marker) :].strip()

    # Remove any remaining memory markers and their content
    for marker in ["MEMORY:", "STORE IN MEMORY:", "REMEMBER:"]:
        if marker in response:
            parts = response.split(marker, 1)
            if len(parts) > 1:
                second_part = parts[1].split("\n\n", 1)
                if len(second_part) > 1:
                    response = parts[0] + second_part[1]
                else:
                    response = parts[0]

    return response.strip()


if __name__ == "__main__":
    run_interactive_health_coach()


🏃‍♂️ WELCOME TO YOUR PERSONAL HEALTH COACH 🏋️‍♀️
I'm your AI health coach with memory. I can provide personalized fitness advice
and will remember details about your fitness journey between conversations.

Type 'exit', 'quit', or 'bye' to end our conversation.


🔍 Searching memory for relevant context...
📭 No previous memories found.

⏳ Thinking...

💾 Storing new information in memory...
✅ Stored: User shared: hi my name is pappu yadav

Coach: Hi Pappu Yadav! 👋 Welcome! I'm so glad you've decided to start your health and fitness journey with me.

Since we're just getting started, I'd love to learn more about you so I can give you the best personalized guidance. Here are a few questions to kick things off:

🎯 **Your Goals:**
- What are you hoping to achieve? (e.g., weight loss, muscle gain, better endurance, general fitness, healthier eating)

📊 **Your Starting Point:**
- How would you describe your current activity level? (sedentary, lightly active, moderately active, very active)
- D